# Módulo 8 · Clase 6 — Regularización: Dropout, Batch Norm, L1/L2

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Técnica | ¿Para qué sirve? |
|---|---------|------------------|
| 1 | El problema: overfitting | Diagnosticar cuándo la red memoriza |
| 2 | L1 y L2 | Penalizar pesos grandes |
| 3 | Dropout | Apagar neuronas aleatoriamente |
| 4 | Batch Normalization | Estabilizar activaciones por capa |
| 5 | Early Stopping | Parar cuando deja de mejorar |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42)
np.random.seed(42)
print(f"TensorFlow: {tf.__version__}")

---
## 1. El problema: overfitting

In [ ]:
# Provocar overfitting deliberadamente: pocos datos, red grande
(X_tr_raw, y_tr_raw), (X_te_raw, y_te_raw) = keras.datasets.mnist.load_data()

# Solo 1000 muestras para entrenar → facilita el overfitting
N_TRAIN = 1000
X_tr_m = X_tr_raw[:N_TRAIN].reshape(-1, 784).astype('float32') / 255.0
y_tr_m = y_tr_raw[:N_TRAIN]
X_te_m = X_te_raw.reshape(-1, 784).astype('float32') / 255.0
y_te_m = y_te_raw

print(f"Train pequeño: {X_tr_m.shape}  Test: {X_te_m.shape}")

def crear_modelo_base(dropout_rate=0.0, l2=0.0, batch_norm=False):
    """Crea el modelo con distintas combinaciones de regularización."""
    reg = regularizers.l2(l2) if l2 > 0 else None
    capas = [layers.Input(shape=(784,))]
    for n in [512, 256, 128]:
        capas.append(layers.Dense(n, activation='relu', kernel_regularizer=reg))
        if batch_norm:
            capas.append(layers.BatchNormalization())
        if dropout_rate > 0:
            capas.append(layers.Dropout(dropout_rate))
    capas.append(layers.Dense(10, activation='softmax'))

    m = keras.Sequential(capas)
    m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
    return m

# Modelo sin regularización (referencia)
tf.random.set_seed(42)
m_base = crear_modelo_base()
h_base = m_base.fit(X_tr_m, y_tr_m, epochs=50, batch_size=64,
                    validation_split=0.15, verbose=0)
_, acc_base_te = m_base.evaluate(X_te_m, y_te_m, verbose=0)

acc_base_tr  = h_base.history['accuracy'][-1]
acc_base_val = h_base.history['val_accuracy'][-1]
print(f"Sin regularización → Train: {acc_base_tr:.4f}, Val: {acc_base_val:.4f}, Test: {acc_base_te:.4f}")
print(f"Gap train-val: {acc_base_tr - acc_base_val:.4f}  ← overfitting")

---
## 2. L1 y L2: penalizar pesos grandes

```
Sin regularización:  Loss = L(y, ŷ)
Con L2 (Ridge):      Loss = L(y, ŷ) + λ·Σwᵢ²     ← penaliza pesos grandes
Con L1 (Lasso):      Loss = L(y, ŷ) + λ·Σ|wᵢ|    ← favorece pesos = 0 (sparse)
Con L1+L2 (ElasticNet): combina ambas

λ (lambda): fuerza de regularización
  λ muy pequeño → poca regularización
  λ muy grande  → pesos → 0, underfitting
  Típico: 1e-4 – 1e-2

En Keras:
  layers.Dense(n, kernel_regularizer=regularizers.l2(0.001))
  layers.Dense(n, kernel_regularizer=regularizers.l1(0.001))
  layers.Dense(n, kernel_regularizer=regularizers.l1_l2(l1=0.001, l2=0.001))
```

In [ ]:
# Comparar L2 con distintos valores de λ
resultados_l2 = {'base': {'hist': h_base, 'acc_te': acc_base_te}}

for lam in [0.0001, 0.001, 0.01]:
    tf.random.set_seed(42)
    m = crear_modelo_base(l2=lam)
    h = m.fit(X_tr_m, y_tr_m, epochs=50, batch_size=64,
              validation_split=0.15, verbose=0)
    _, acc_te = m.evaluate(X_te_m, y_te_m, verbose=0)
    resultados_l2[f'L2={lam}'] = {'hist': h, 'acc_te': acc_te}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colores = ['gray', 'steelblue', 'darkorange', 'green']
for (nombre, res), color in zip(resultados_l2.items(), colores):
    acc_tr  = res['hist'].history['accuracy'][-1]
    acc_val = res['hist'].history['val_accuracy'][-1]
    label = f"{nombre} (tr={acc_tr:.3f}, val={acc_val:.3f}, te={res['acc_te']:.3f})"
    axes[0].plot(res['hist'].history['val_loss'],     color=color, lw=2, label=nombre)
    axes[1].plot(res['hist'].history['val_accuracy'], color=color, lw=2, label=label)

for ax in axes:
    ax.set_xlabel('Época'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
axes[0].set_title('Val Loss con L2'); axes[1].set_title('Val Accuracy con L2')
plt.tight_layout(); plt.show()

---
## 3. Dropout: apagar neuronas aleatoriamente

```
Durante el entrenamiento, cada neurona se 'apaga' con probabilidad p (dropout_rate).
Las neuronas activas se escalan ×(1/(1-p)) para compensar.

Durante la inferencia (predict/evaluate), se usan TODAS las neuronas sin aleatoridad.

Efecto:
  La red aprende representaciones redundantes y robustas.
  Equivale a entrenar un 'ensemble' de 2^n subredes distintas.
  Muy efectivo para evitar coadaptación entre neuronas.

Rate típicos:
  0.2 – 0.3  para capas de entrada
  0.4 – 0.5  para capas ocultas intermedias
  >0.6       raramente útil

En Keras:
  layers.Dropout(0.4)   # va después de la capa Dense, antes de la siguiente
```

In [ ]:
# Visualizar qué hace Dropout
np.random.seed(0)
n_neuronas = 20
activaciones = np.random.rand(n_neuronas)

fig, axes = plt.subplots(1, 3, figsize=(14, 3))

# Sin dropout
axes[0].bar(range(n_neuronas), activaciones, color='steelblue')
axes[0].set_title('Sin Dropout (todas activas)')

# Con dropout rate=0.5
mask = np.random.rand(n_neuronas) > 0.5
act_drop = activaciones * mask / 0.5  # escalar
colores_d = ['steelblue' if m else 'lightgray' for m in mask]
axes[1].bar(range(n_neuronas), act_drop, color=colores_d)
axes[1].set_title(f'Con Dropout(0.5) — {mask.sum()} activas, {(~mask).sum()} apagadas')

# Distinto paso (distinta máscara aleatoria)
mask2 = np.random.rand(n_neuronas) > 0.5
act_drop2 = activaciones * mask2 / 0.5
colores_d2 = ['steelblue' if m else 'lightgray' for m in mask2]
axes[2].bar(range(n_neuronas), act_drop2, color=colores_d2)
axes[2].set_title(f'Siguiente paso — {mask2.sum()} activas (máscara distinta)')

for ax in axes:
    ax.set_xlabel('Neurona'); ax.set_ylabel('Activación')
plt.tight_layout(); plt.show()
print("→ Cada mini-batch usa una subred diferente → ensemble implícito")

In [ ]:
# Comparar distintas tasas de Dropout
resultados_drop = {}
for rate in [0.0, 0.2, 0.4, 0.6]:
    tf.random.set_seed(42)
    m = crear_modelo_base(dropout_rate=rate)
    h = m.fit(X_tr_m, y_tr_m, epochs=60, batch_size=64,
              validation_split=0.15, verbose=0)
    _, acc_te = m.evaluate(X_te_m, y_te_m, verbose=0)
    acc_tr  = h.history['accuracy'][-1]
    acc_val = h.history['val_accuracy'][-1]
    resultados_drop[f'drop={rate}'] = {'hist': h, 'acc_te': acc_te,
                                        'acc_tr': acc_tr, 'acc_val': acc_val}
    print(f"Dropout={rate}: tr={acc_tr:.3f}, val={acc_val:.3f}, te={acc_te:.3f}, gap={acc_tr-acc_val:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colores = ['gray', 'steelblue', 'darkorange', 'green']
for (nombre, res), color in zip(resultados_drop.items(), colores):
    axes[0].plot(res['hist'].history['val_loss'],     color=color, lw=2, label=nombre)
    axes[1].plot(res['hist'].history['val_accuracy'], color=color, lw=2, label=nombre)
for ax in axes:
    ax.set_xlabel('Época'); ax.legend(); ax.grid(True, alpha=0.3)
axes[0].set_title('Val Loss'); axes[1].set_title('Val Accuracy')
plt.tight_layout(); plt.show()

---
## 4. Batch Normalization: estabilizar activaciones

```
Problema: las activaciones de cada capa pueden tener distribuciones muy distintas
          y cambiar mucho durante el entrenamiento (internal covariate shift).

Batch Normalization (Ioffe & Szegedy, 2015):
  Por cada mini-batch, normaliza la salida de la capa anterior:
    μ = mean(z)   σ² = var(z)
    z_norm = (z - μ) / sqrt(σ² + ε)
    out = γ · z_norm + β    ← γ y β son parámetros aprendibles

  Durante inferencia usa media/varianza móviles calculadas en el entrenamiento.

Ventajas:
  Permite learning rates más altos → entrenamiento más rápido
  Reduce dependencia de la inicialización
  Tiene efecto regularizador ligero (no reemplaza a Dropout)

Posición típica: Dense → BatchNorm → Activation (en algunas arquitecturas)
              o: Dense(activation='relu') → BatchNorm  (la más común en Keras)

En Keras:
  layers.BatchNormalization()
```

In [ ]:
# Visualizar qué hace BatchNorm en las activaciones
np.random.seed(42)
activaciones_sin_bn = np.random.exponential(scale=5, size=(1000,)) - 2  # distribución asimétrica

# BN normaliza
mu  = activaciones_sin_bn.mean()
std = activaciones_sin_bn.std() + 1e-8
activaciones_con_bn = (activaciones_sin_bn - mu) / std

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(activaciones_sin_bn, bins=50, color='coral', edgecolor='white')
axes[0].set_title(f'Sin BatchNorm: μ={mu:.2f}, σ={std:.2f}')
axes[0].axvline(0, color='black', ls='--')
axes[0].set_xlabel('Activación')

axes[1].hist(activaciones_con_bn, bins=50, color='steelblue', edgecolor='white')
axes[1].set_title(f'Con BatchNorm: μ≈{activaciones_con_bn.mean():.2f}, σ≈{activaciones_con_bn.std():.2f}')
axes[1].axvline(0, color='black', ls='--')
axes[1].set_xlabel('Activación normalizada')

plt.suptitle('Efecto de Batch Normalization', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Comparar: sin BN, con BN, con BN+Dropout
configs = [
    ('Sin regularización',  dict(dropout_rate=0.0, l2=0.0, batch_norm=False)),
    ('BatchNorm',           dict(dropout_rate=0.0, l2=0.0, batch_norm=True)),
    ('Dropout(0.4)',        dict(dropout_rate=0.4, l2=0.0, batch_norm=False)),
    ('BN + Dropout(0.3)',   dict(dropout_rate=0.3, l2=0.0, batch_norm=True)),
]

print(f"{'Config':25} {'Train':>8} {'Val':>8} {'Test':>8} {'Gap':>8}")
print('-' * 60)
historiales = {}
for nombre, kwargs in configs:
    tf.random.set_seed(42)
    m = crear_modelo_base(**kwargs)
    h = m.fit(X_tr_m, y_tr_m, epochs=50, batch_size=64,
              validation_split=0.15, verbose=0)
    _, acc_te = m.evaluate(X_te_m, y_te_m, verbose=0)
    acc_tr  = h.history['accuracy'][-1]
    acc_val = h.history['val_accuracy'][-1]
    historiales[nombre] = h
    print(f"{nombre:25} {acc_tr:8.4f} {acc_val:8.4f} {acc_te:8.4f} {acc_tr-acc_val:8.4f}")

---
## 5. Early Stopping: parar en el momento óptimo

In [ ]:
# Visualizar cuándo el modelo 'empieza a memorizar'
tf.random.set_seed(42)
m_es = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
m_es.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

# Entrenar sin EarlyStopping hasta que se vea el overfitting
h_sin_es = m_es.fit(X_tr_m, y_tr_m, epochs=100, batch_size=64,
                    validation_split=0.15, verbose=0)

val_loss = np.array(h_sin_es.history['val_loss'])
mejor_epoca = val_loss.argmin()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(h_sin_es.history['loss'],     'steelblue', lw=2, label='Train')
axes[0].plot(h_sin_es.history['val_loss'], 'coral',     lw=2, label='Val')
axes[0].axvline(mejor_epoca, color='green', ls='--', lw=2,
                label=f'Mejor época ({mejor_epoca+1})')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
axes[0].set_title('Sin Early Stopping'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(h_sin_es.history['accuracy'],     'steelblue', lw=2, label='Train')
axes[1].plot(h_sin_es.history['val_accuracy'], 'coral',     lw=2, label='Val')
axes[1].axvline(mejor_epoca, color='green', ls='--', lw=2, label=f'Parar aquí')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Sin Early Stopping'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Val loss mínimo en época {mejor_epoca+1} de {len(val_loss)}")
print(f"→ EarlyStopping(patience=10) habría parado ~en época {mejor_epoca+11}")

In [ ]:
# Estrategia combinada: la mejor práctica
tf.random.set_seed(42)
modelo_final = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(5e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(5e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax'),
])

modelo_final.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

h_final = modelo_final.fit(
    X_tr_m, y_tr_m,
    epochs=200, batch_size=64,
    validation_split=0.15,
    callbacks=[
        keras.callbacks.EarlyStopping('val_loss', patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=8, min_lr=1e-6),
    ],
    verbose=0
)

_, acc_te_final = modelo_final.evaluate(X_te_m, y_te_m, verbose=0)
acc_tr_final  = h_final.history['accuracy'][-1]
acc_val_final = h_final.history['val_accuracy'][-1]
n_epocas = len(h_final.history['loss'])

print(f"Con L2+BatchNorm+Dropout+EarlyStopping ({n_epocas} épocas):")
print(f"  Train: {acc_tr_final:.4f}  Val: {acc_val_final:.4f}  Test: {acc_te_final:.4f}")
print(f"  Gap train-val: {acc_tr_final-acc_val_final:.4f}")

---
## Práctica

### Ejercicio 1 — Regularización en Fashion MNIST

**Enunciado:** Fashion MNIST es más difícil que MNIST y más propenso al overfitting.

1. Usa solo 2000 muestras para entrenar
2. Compara: sin regularización vs Dropout(0.4) vs L2(0.001)
3. ¿Cuál generaliza mejor en test (todos los 10000)?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
N = 2000
X_tr_n = X_tr[:N].reshape(-1, 784).astype('float32') / 255.0
y_tr_n = y_tr[:N]
X_te_n = X_te.reshape(-1, 784).astype('float32') / 255.0

def modelo_fashion(dropout=0.0, l2=0.0):
    reg = regularizers.l2(l2) if l2 > 0 else None
    capas = [layers.Input(shape=(784,))]
    for n in [256, 128]:
        capas.append(layers.Dense(n, activation='relu', kernel_regularizer=reg))
        if dropout > 0:
            capas.append(layers.Dropout(dropout))
    capas.append(layers.Dense(10, activation='softmax'))
    m = keras.Sequential(capas)
    m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
    return m

configs = [
    ('Sin regularización', {}),
    ('Dropout(0.4)',       {'dropout': 0.4}),
    ('L2(0.001)',          {'l2': 0.001}),
]

print(f"{'Config':22} {'Train':>8} {'Val':>8} {'Test':>8}")
print('-' * 50)
for nombre, kwargs in configs:
    tf.random.set_seed(42)
    m = modelo_fashion(**kwargs)
    h = m.fit(X_tr_n, y_tr_n, epochs=60, batch_size=64,
              validation_split=0.15, verbose=0)
    _, acc_te = m.evaluate(X_te_n, y_te, verbose=0)
    acc_tr  = h.history['accuracy'][-1]
    acc_val = h.history['val_accuracy'][-1]
    print(f"{nombre:22} {acc_tr:8.4f} {acc_val:8.4f} {acc_te:8.4f}")
```
</details>

---
### Ejercicio 2 — Dropout en test vs train

**Enunciado:** Dropout se comporta diferente en train y test. Verifica esto:

1. Toma un modelo con Dropout(0.5) ya entrenado
2. Usa `tf.keras.backend.learning_phase()` para activar el modo train
3. Haz 10 predicciones distintas sobre la misma muestra con `training=True`
4. Observa que las predicciones cambian cada vez (Monte Carlo Dropout)
5. Promedia las 10 predicciones → ¿mejora respecto a una sola predicción?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax'),
])
m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
m.fit(X_tr, y_tr, epochs=10, batch_size=128, verbose=0)

# Tomar una imagen
img = X_te[0:1]  # dígito 7
n_muestras = 20

# Monte Carlo Dropout: predicciones con dropout activo
predicciones_mc = np.array([
    m(img, training=True).numpy()[0]  # training=True activa Dropout
    for _ in range(n_muestras)
])

pred_normal = m.predict(img, verbose=0)[0]  # training=False (sin dropout)
pred_mc     = predicciones_mc.mean(axis=0)  # promedio Monte Carlo

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = range(10)

# Varianza en cada clase
axes[0].bar(x, predicciones_mc.mean(axis=0), yerr=predicciones_mc.std(axis=0),
            color='steelblue', capsize=5, label='Media ± std MC')
axes[0].bar(x, pred_normal, alpha=0.4, color='coral', label='Predicción normal')
axes[0].set_xticks(x); axes[0].set_xlabel('Clase (dígito)')
axes[0].set_title('Monte Carlo Dropout — incertidumbre por clase')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

# Variación entre predicciones
for i, pred in enumerate(predicciones_mc):
    axes[1].plot(x, pred, 'o-', alpha=0.4, lw=0.8)
axes[1].plot(x, pred_mc, 'k-o', lw=2.5, markersize=7, label='Promedio MC')
axes[1].plot(x, pred_normal, 'r--s', lw=2, label='Sin dropout (normal)')
axes[1].set_xticks(x); axes[1].set_xlabel('Clase')
axes[1].set_title('20 predicciones MC distintas')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f"Predicción normal: clase {pred_normal.argmax()} ({pred_normal.max():.4f})")
print(f"Promedio MC ({n_muestras} muestras): clase {pred_mc.argmax()} ({pred_mc.max():.4f})")
print(f"Clase real: {y_te[0]}")
print(f"\nIncertidumbre (std): {predicciones_mc.std(axis=0).round(4)}")
```
</details>

---
### Ejercicio 3 — BatchNorm: velocidad de convergencia

**Enunciado:** BatchNorm permite usar learning rates más altos.

1. Entrena dos modelos idénticos (256 → 128 → 10) en MNIST completo:
   - Sin BatchNorm, lr=0.01
   - Con BatchNorm, lr=0.01
2. Compara cuál converge más rápido en las primeras 10 épocas
3. ¿Qué pasa sin BatchNorm con lr alto?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

configs = [
    ('Sin BN, lr=0.01',  False, 0.01),
    ('Con BN, lr=0.01',  True,  0.01),
    ('Sin BN, lr=0.001', False, 0.001),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colores = ['coral', 'steelblue', 'gray']

for (nombre, bn, lr), color in zip(configs, colores):
    tf.random.set_seed(42)
    capas = [layers.Input(shape=(784,))]
    for n in [256, 128]:
        capas.append(layers.Dense(n, activation='relu'))
        if bn:
            capas.append(layers.BatchNormalization())
    capas.append(layers.Dense(10, activation='softmax'))
    m = keras.Sequential(capas)
    m.compile(keras.optimizers.Adam(lr), 'sparse_categorical_crossentropy', ['accuracy'])
    h = m.fit(X_tr, y_tr, epochs=10, batch_size=128, validation_split=0.1, verbose=0)

    axes[0].plot(h.history['loss'],     color=color, lw=2, label=nombre)
    axes[1].plot(h.history['val_accuracy'], color=color, lw=2, label=nombre)

for ax in axes:
    ax.set_xlabel('Época'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
axes[0].set_title('Train Loss — primeras 10 épocas')
axes[1].set_title('Val Accuracy — primeras 10 épocas')
plt.tight_layout(); plt.show()

print("→ Con BatchNorm, lr=0.01 converge más rápido que sin BN con la misma lr.")
print("  Sin BN y lr alta, el entrenamiento puede ser inestable.")
```
</details>

---
### Ejercicio 4 — Búsqueda de hiperparámetros de regularización

**Enunciado:** Encuentra la combinación óptima:

1. Dataset: Fashion MNIST completo
2. Prueba todas las combinaciones de:
   - dropout_rate: [0.1, 0.3, 0.5]
   - l2: [0.0, 0.0005, 0.001]
3. Cada modelo: 3 épocas (rápido), evalúa en val
4. Muestra el ranking de configuraciones por val_accuracy

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import numpy as np

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

dropout_rates = [0.1, 0.3, 0.5]
l2_vals      = [0.0, 0.0005, 0.001]

resultados = []
for dr in dropout_rates:
    for l2 in l2_vals:
        reg = regularizers.l2(l2) if l2 > 0 else None
        tf.random.set_seed(42)
        m = keras.Sequential([
            layers.Input(shape=(784,)),
            layers.Dense(256, activation='relu', kernel_regularizer=reg),
            layers.Dropout(dr),
            layers.Dense(128, activation='relu', kernel_regularizer=reg),
            layers.Dropout(dr),
            layers.Dense(10, activation='softmax'),
        ])
        m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
        h = m.fit(X_tr, y_tr, epochs=5, batch_size=128,
                  validation_split=0.1, verbose=0)
        val_acc = h.history['val_accuracy'][-1]
        resultados.append({'dropout': dr, 'l2': l2, 'val_acc': val_acc})
        print(f"dropout={dr}, l2={l2:.4f} → val_acc={val_acc:.4f}")

resultados.sort(key=lambda x: -x['val_acc'])
print("\n== Ranking ==")
for i, r in enumerate(resultados[:5], 1):
    print(f"{i}. dropout={r['dropout']}, l2={r['l2']:.4f} → {r['val_acc']:.4f}")
```
</details>

---
### Ejercicio 5 (reto) — Regularización personalizada

**Enunciado:** Implementa un callback personalizado que monitorice el gap train-val y active Dropout dinámicamente:

1. Crea una clase `GapMonitor(keras.callbacks.Callback)` que registre `train_acc - val_acc` en cada época
2. Al finalizar el entrenamiento, plóta el gap por época
3. Señala en qué época el gap supera 0.05 (inicio del overfitting)
4. Usa este callback en el entrenamiento de Fashion MNIST con 5000 muestras

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

class GapMonitor(keras.callbacks.Callback):
    def __init__(self, umbral=0.05):
        super().__init__()
        self.umbral   = umbral
        self.gaps     = []
        self.alerta   = None

    def on_epoch_end(self, epoch, logs=None):
        tr  = logs.get('accuracy',     0)
        val = logs.get('val_accuracy', 0)
        gap = tr - val
        self.gaps.append(gap)
        if self.alerta is None and gap > self.umbral:
            self.alerta = epoch
            print(f"  [GapMonitor] Overfitting detectado en época {epoch+1}: gap={gap:.4f}")

    def plot(self):
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(self.gaps, 'steelblue', lw=2)
        ax.axhline(self.umbral, color='red', ls='--', label=f'Umbral ({self.umbral})')
        if self.alerta is not None:
            ax.axvline(self.alerta, color='orange', ls='--', label=f'Alerta época {self.alerta+1}')
        ax.set_xlabel('Época'); ax.set_ylabel('Gap (train - val accuracy)')
        ax.set_title('Monitorización del gap train-val'); ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr[:5000].reshape(-1, 784).astype('float32') / 255.0
y_tr = y_tr[:5000]
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

monitor = GapMonitor(umbral=0.05)
m.fit(X_tr, y_tr, epochs=60, batch_size=64, validation_split=0.15,
      callbacks=[monitor], verbose=0)

monitor.plot()
_, acc = m.evaluate(X_te, y_te, verbose=0)
print(f"Test accuracy: {acc:.4f}")
```
</details>

---
## Resumen

```python
from tensorflow.keras import regularizers

# L2 (weight decay)
layers.Dense(n, activation='relu', kernel_regularizer=regularizers.l2(0.001))

# Dropout: va después de la capa densa
layers.Dense(n, activation='relu')
layers.Dropout(0.4)   # apaga 40% de neuronas en cada paso de train

# Batch Normalization: va después de Dense, antes/después de activación
layers.Dense(n, activation='relu')
layers.BatchNormalization()

# Early Stopping
keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)

# Estrategia recomendada (en orden de prioridad):
# 1. EarlyStopping siempre
# 2. Dropout si hay overfitting claro
# 3. BatchNorm si el entrenamiento es lento o inestable
# 4. L2 como complemento, no como sustituto
```

| Técnica | Cuándo usarla | Parámetro clave |
|---------|---------------|-----------------|
| L2 | Siempre disponible, complemento suave | λ ∈ [1e-4, 1e-2] |
| Dropout | Overfitting fuerte, redes densas | rate ∈ [0.2, 0.5] |
| BatchNorm | Entrenar más rápido, redes profundas | — (sin hiperparámetros críticos) |
| EarlyStopping | Siempre | patience ∈ [5, 20] |

## ¿Qué viene en la próxima clase?
En la **Clase 7** aprenderemos **Redes Convolucionales (CNN)** — la arquitectura que revolucionó la visión artificial, capaz de detectar patrones visuales directamente en los pixels.